Pipeline : Automatic & Reusable Data Preprocessing Pipeline (Scikit-learn | No Models).        
Author : Mir Musaib.


📌 Objective

Build a robust, reusable preprocessing pipeline that automatically:

1.Detects feature types
2.Applies appropriate transformations
3.Converts raw tabular data into ML-ready numerical format
4.Avoids data leakage and manual repetition

🧠 Why an Automatic Pipeline?

Manual preprocessing:

1.Is error-prone.        
2.Breaks consistency.         
3.Doesn’t scale.       

A pipeline:

1.Ensures same preprocessing every time.        
2.Is reusable across datasets.              
3.Matches industry standards.       

📥 Step 1: Import Required Libraries

In [43]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

🔎 Step 2: ## Dataset Loading

The raw dataset is loaded using pandas.  
At this stage, no transformations are applied. The purpose is to inspect the dataset structure, column names, and dimensions before preprocessing.

In [44]:
df = pd.read_csv(
    "/workspaces/Data_Science_Internship/Week_4/Dataset/Customer Segmentation.csv"
)

print("Initial shape:", df.shape)
print("Columns:", df.columns.tolist())

Initial shape: (51000, 23)
Columns: ['first_name', 'last_name', 'title', 'gender', 'email', 'city', 'country', 'country_code', 'latitude', 'longitude', 'phone', 'street_address', 'street_name', 'street_number', 'street_suffix', 'time_zone', 'company_name', 'department', 'job_title', 'language', 'university', 'linkedin_skill', 'ip_address']


🏗️ Step 3: ## Identifier and Metadata Columns

Identifier and metadata columns (such as IDs) do not carry predictive information for machine learning models.  
However, they are important for **traceability and reference**.

Therefore:
- These columns are **excluded from preprocessing**
- They are **preserved unchanged** in the final output using passthrough

In [45]:
def get_metadata_columns(df):
    return [col for col in df.columns if 'id' in col.lower()]

metadata_cols = get_metadata_columns(df)

print("Metadata columns (kept as-is):", metadata_cols)

Metadata columns (kept as-is): []


▶️ Step 4: ## Automatic Feature Type Detection

To make the pipeline reusable across datasets, feature types are detected automatically:

- **Numerical features** → scaled
- **Binary categorical features** → encoded efficiently
- **Nominal categorical features** → one-hot encoded

Identifier columns are explicitly excluded from this detection process to prevent unintended transformations.

In [46]:
def detect_feature_types(df, exclude_cols):
    work_df = df.drop(columns=exclude_cols)

    numeric_features = work_df.select_dtypes(
        include=['int64', 'float64']
    ).columns.tolist()

    categorical_features = work_df.select_dtypes(
        include=['object', 'category', 'string']
    ).columns.tolist()

    binary_features = [c for c in categorical_features if work_df[c].nunique() == 2]
    nominal_features = [c for c in categorical_features if work_df[c].nunique() > 2]

    return numeric_features, binary_features, nominal_features

## Preprocessing Pipeline Construction

A `ColumnTransformer` is used to apply different preprocessing steps to different feature types:

- Numerical features are standardized
- Binary categorical features are encoded safely
- Nominal categorical features are one-hot encoded
- Identifier columns are passed through unchanged

This ensures consistent and reproducible preprocessing.

In [47]:
def build_auto_preprocessor(df):
    metadata_cols = get_metadata_columns(df)

    num_features, bin_features, nom_features = detect_feature_types(
        df, exclude_cols=metadata_cols
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_features),
            ('bin', OneHotEncoder(drop='if_binary', handle_unknown='ignore'), bin_features),
            ('nom', OneHotEncoder(drop='first', handle_unknown='ignore'), nom_features)
        ],
        remainder='passthrough'   # 👈 keeps ID / metadata columns
    )

    return preprocessor

## Applying the Preprocessing Pipeline

The pipeline is fitted on the raw dataset and used to transform the data into a
numerical, ML-ready format.

In [48]:
auto_preprocessor = build_auto_preprocessor(df)

X_processed = auto_preprocessor.fit_transform(df)

print("Processed array shape:", X_processed.shape)

Processed array shape: (51000, 282670)


## Extracting Feature Names

Feature names are extracted from the fitted pipeline to preserve interpretability
and track feature origins.

In [49]:
feature_names = auto_preprocessor.get_feature_names_out()

processed_df = pd.DataFrame.sparse.from_spmatrix(
    X_processed,
    columns=feature_names
)

processed_df.to_csv("processed_data_with_ids.csv", index=False)

KeyboardInterrupt: 

## Verifying Identifier Preservation

This step confirms that identifier columns were preserved unchanged using passthrough.

In [ ]:
[id_col for id_col in processed_df.columns if 'id' in id_col.lower()]

[]

## Saving the Processed Dataset

The final ML-ready dataset is saved as a CSV file for reuse in downstream tasks.

In [ ]:
processed_df.to_csv("processed_data_with_ids.csv", index=False)

print("✅ Final processed dataset saved with IDs")

✅ Final processed dataset saved with IDs


## Conclusion

An automated preprocessing pipeline was successfully implemented to:
- Detect feature types dynamically
- Apply appropriate transformations
- Preserve identifier columns
- Produce a clean, ML-ready dataset

This approach ensures scalability, reproducibility, and industry-standard preprocessing.